# DiT-Based Anomaly Detection â€” Comprehensive Analysis
**Group 6 | DATA-MSML 612 | University of Maryland**

Produces all results for the interim report:
- **Section 0**: Environment setup (GPU, Drive, repo, packages, globals)
- **Section 1**: Data download + checkpoint restore
- **Section 2**: DiT evaluation â€” L2 scoring, all 15 categories *(primary results)*
- **Section 3**: T_partial ablation sweep
- **Section 4**: PatchCore baseline â€” all 15 categories *(SOTA comparison)*
- **Section 5**: All report figures
- **Section 6**: Summary tables + LaTeX
- **Section 7**: Package to Drive + verify

**Prerequisites:** ALL_OUTPUT.zip in Drive root (contains all 15 DiT + 1 UNet checkpoints).

## Section 0 â€” Environment Setup

In [ ]:
import subprocess, sys, torch

r = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
print('GPU :', r.stdout.strip() or 'NO GPU â€” connect a T4 runtime before continuing')
print('Python :', sys.version[:40])
print('PyTorch:', torch.__version__)
print('CUDA   :', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('WARNING: No GPU detected. Training/eval will be very slow.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print('Drive mounted.')

import os, subprocess

# â”€â”€ Update this URL if your repo is at a different address â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
REPO_URL = '<YOUR_REPO_URL>.git'
REPO_DIR = '/content/dit_anomaly'

if not os.path.exists(REPO_DIR):
    r = subprocess.run(['git', 'clone', REPO_URL, REPO_DIR],
                       capture_output=True, text=True)
    print((r.stdout + r.stderr)[-400:])
else:
    r = subprocess.run(['git', 'pull', '--rebase'],
                       capture_output=True, text=True, cwd=REPO_DIR)
    print('Repo:', (r.stdout + r.stderr)[-200:].strip())

CODE_DIR = os.path.join(REPO_DIR, 'code')
os.chdir(CODE_DIR)
print('Working dir:', os.getcwd())

In [ ]:
import subprocess, sys, torch

# Install packages not present in base Colab image
_deps = ['einops', 'timm', 'pytorch-msssim', 'scikit-image', 'lpips']
r = subprocess.run(['pip', 'install', '-q'] + _deps, capture_output=True, text=True)
if r.returncode != 0:
    print('pip warning:', r.stderr[-300:])
else:
    print('Packages ready:', _deps)

# Add code/ to import path so 'from src.xxx import yyy' works
sys.path.insert(0, '.')

# â”€â”€ Global constants used by every subsequent cell â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_ROOT = '../data/mvtec'
ALL_CATEGORIES = [
    'bottle', 'cable', 'capsule', 'carpet', 'grid',
    'hazelnut', 'leather', 'metal_nut', 'pill', 'screw',
    'tile', 'toothbrush', 'transistor', 'wood', 'zipper',
]
print('Device:', DEVICE, '| Categories:', len(ALL_CATEGORIES))

## Section 1 â€” Data & Checkpoints

In [ ]:
import os, json
from pathlib import Path

def _write_kaggle_json(username, key):
    p = Path('/root/.config/kaggle/kaggle.json')
    p.parent.mkdir(parents=True, exist_ok=True)
    # Kaggle CLI requires exactly these two field names: 'username' and 'key'
    p.write_text(json.dumps({'username': username, 'key': key}))
    p.chmod(0o600)

_ok = False
try:
    from google.colab import userdata
    _u = userdata.get('KAGGLE_USERNAME')
    _k = userdata.get('KAGGLE_KEY')
    if _u and _k:
        _write_kaggle_json(_u, _k)
        print('Kaggle credentials set from Colab Secrets. Username:', _u)
        _ok = True
    else:
        raise ValueError('Secrets missing or empty')
except Exception as _e:
    print('Colab Secrets unavailable ({}) -- trying .env'.format(_e))

if not _ok:
    _env = Path('.env')
    if _env.exists():
        for _line in _env.read_text().splitlines():
            if '=' in _line and not _line.startswith('#'):
                _kk, _vv = _line.split('=', 1)
                os.environ[_kk.strip()] = _vv.strip()
    _u = os.environ.get('KAGGLE_USERNAME', '')
    _k = os.environ.get('KAGGLE_KEY', '')
    if _u and _k:
        _write_kaggle_json(_u, _k)
        print('Credentials set from .env. Username:', _u)
        _ok = True

if not _ok:
    raise RuntimeError(
        'No Kaggle credentials found.\n'
        'Fix: open Colab Secrets (key icon in left sidebar), '
        'add KAGGLE_USERNAME and KAGGLE_KEY.'
    )

import subprocess
_r = subprocess.run(['kaggle', 'config', 'view'], capture_output=True, text=True)
print(_r.stdout.strip() if _r.returncode == 0 else 'Auth error: ' + _r.stderr[:150])

In [ ]:
import os, zipfile, shutil, subprocess
from pathlib import Path

DATASET_SLUG = 'ipythonx/mvtec-ad'   # terms accepted; full MVTec AD dataset
DL_DIR       = '/tmp/mvtec_dl'
EXTRACT_DIR  = '/tmp/mvtec_extract'

os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(DL_DIR,    exist_ok=True)

missing = [c for c in ALL_CATEGORIES
           if not Path('{}/{}/train/good'.format(DATA_ROOT, c)).exists()]

if not missing:
    print('All 15 categories already present. Skipping download.')
else:
    print('{} categories missing: {}'.format(len(missing), missing))
    print('Downloading {} (~4.9 GB). Expect 8-12 min on Colab...'.format(DATASET_SLUG))

    _r = subprocess.run(
        ['kaggle', 'datasets', 'download', '-d', DATASET_SLUG, '-p', DL_DIR, '-q'],
        capture_output=True, text=True
    )
    if _r.returncode != 0:
        raise RuntimeError('kaggle download failed:\n' + _r.stderr)

    _zips = list(Path(DL_DIR).glob('*.zip'))
    if not _zips:
        raise RuntimeError('No zip found in {}. Contents: {}'.format(
            DL_DIR, [p.name for p in Path(DL_DIR).iterdir()]))

    _zip_path = _zips[0]
    print('Downloaded: {} ({:.2f} GB). Extracting...'.format(
        _zip_path.name, _zip_path.stat().st_size / 1e9))

    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(_zip_path, 'r') as _z:
        _z.extractall(EXTRACT_DIR)
    _zip_path.unlink()
    print('Extraction complete.')

    # Auto-detect root: handles flat layout AND nested (e.g. mvtec_anomaly_detection/)
    _hits = list(Path(EXTRACT_DIR).rglob('bottle/train/good'))
    if not _hits:
        raise RuntimeError('Cannot find bottle/train/good after extraction. '
                           'Top-level: {}'.format([p.name for p in Path(EXTRACT_DIR).iterdir()]))

    _data_src = _hits[0].parent.parent.parent   # parent of 'bottle/'
    print('Dataset root detected at:', _data_src)

    for _cat in missing:
        _src = _data_src / _cat
        _dst = Path(DATA_ROOT) / _cat
        if not _src.exists():
            print('WARNING: {} not found in extracted data'.format(_cat))
            continue
        if _dst.exists():
            shutil.rmtree(_dst)
        shutil.move(str(_src), str(_dst))
        print('  Moved:', _cat)

    shutil.rmtree(EXTRACT_DIR, ignore_errors=True)

# Verify
_present = [c for c in ALL_CATEGORIES
            if Path('{}/{}/train/good'.format(DATA_ROOT, c)).exists()]
print('\nVerified: {}/15 categories.'.format(len(_present)))
if _present:
    _ex = _present[0]
    _n = len(list(Path('{}/{}/train/good'.format(DATA_ROOT, _ex)).glob('*.png')))
    _splits = [d.name for d in sorted(Path('{}/{}/test'.format(DATA_ROOT, _ex)).iterdir())
               if d.is_dir()]
    print('Spot check ({}): {} train images | test splits: {}'.format(_ex, _n, _splits))
if len(_present) < 15:
    print('Still missing:', [c for c in ALL_CATEGORIES if c not in _present])

In [ ]:
import os, zipfile, shutil
from pathlib import Path

DRIVE_DIR       = '/content/drive/MyDrive'
LOCAL_CKPT_DIT  = 'output/checkpoints'
LOCAL_CKPT_UNET = 'output/checkpoints_unet'
os.makedirs(LOCAL_CKPT_DIT,  exist_ok=True)
os.makedirs(LOCAL_CKPT_UNET, exist_ok=True)

# Show all zips in Drive for transparency
_drive_zips = sorted(Path(DRIVE_DIR).glob('*.zip'))
print('ZIP files in Drive root:')
for _f in _drive_zips:
    print('  {} ({:.1f} MB)'.format(_f.name, _f.stat().st_size / 1e6))
if not _drive_zips:
    print('  (none found)')

_restored_dit  = len(list(Path(LOCAL_CKPT_DIT).rglob('best.pt')))
_restored_unet = len(list(Path(LOCAL_CKPT_UNET).rglob('best.pt')))
print('Already on disk: {} DiT, {} UNet'.format(_restored_dit, _restored_unet))

if _restored_dit < 15:
    # Try zips in priority order; ALL_OUTPUT.zip is the main one
    _candidates = ['dit_anomaly_checkpoints.zip', 'ALL_OUTPUT.zip',
                   'all_outputs.zip', 'all_output.zip']
    for _f in _drive_zips:
        if _f.name not in _candidates:
            _candidates.append(_f.name)

    for _zip_name in _candidates:
        _zip_path = Path(DRIVE_DIR) / _zip_name
        if not _zip_path.exists():
            continue

        with zipfile.ZipFile(_zip_path, 'r') as _z:
            _pt_files = [n for n in _z.namelist() if n.endswith('best.pt')]

        if not _pt_files:
            print('{}: no best.pt files, skipping.'.format(_zip_name))
            continue

        print('\nFound {} checkpoint(s) in {}. Extracting...'.format(
            len(_pt_files), _zip_name))

        with zipfile.ZipFile(_zip_path, 'r') as _z:
            for _pt in _pt_files:
                _parts = Path(_pt).parts
                try:
                    _idx = next(i for i, p in enumerate(_parts) if 'checkpoint' in p.lower())
                except StopIteration:
                    _idx = 0
                _rel = Path(*_parts[_idx:])
                if 'unet' in str(_rel).lower():
                    _dest = Path(LOCAL_CKPT_UNET) / Path(*_rel.parts[1:])
                else:
                    _dest = Path(LOCAL_CKPT_DIT)  / Path(*_rel.parts[1:])
                _dest.parent.mkdir(parents=True, exist_ok=True)
                with _z.open(_pt) as _src, open(_dest, 'wb') as _dst:
                    shutil.copyfileobj(_src, _dst)

        _restored_dit  = len(list(Path(LOCAL_CKPT_DIT).rglob('best.pt')))
        _restored_unet = len(list(Path(LOCAL_CKPT_UNET).rglob('best.pt')))
        print('  DiT: {}/15 | UNet: {}'.format(_restored_dit, _restored_unet))
        if _restored_dit >= 15:
            break
else:
    print('All 15 checkpoints already on disk.')

HAVE_CHECKPOINTS = _restored_dit > 0
print('\nReady: {}/15 DiT | {} UNet | HAVE_CHECKPOINTS={}'.format(
    _restored_dit, _restored_unet, HAVE_CHECKPOINTS))
if _restored_dit > 0:
    _cats = sorted([p.parent.name for p in Path(LOCAL_CKPT_DIT).rglob('best.pt')])
    print('Categories:', _cats)

## Section 2 â€” DiT Evaluation: L2 Scoring (Primary Results)

In [ ]:
import sys, json, os, torch, numpy as np
from pathlib import Path

sys.path.insert(0, '.')

if not HAVE_CHECKPOINTS:
    print('No checkpoints found. Skipping L2 evaluation.')
    print('Re-run cell 1C after verifying ALL_OUTPUT.zip is in Drive root.')
else:
    from src.dataset import get_dataloaders
    from src.diffusion import GaussianDiffusion, cosine_beta_schedule
    from src.dit import DiT_Tiny
    from src.scoring import FeatureExtractor
    from src.evaluate import evaluate_category

    _CKPT_DIR  = 'output/checkpoints'
    _L2_OUT    = 'output/results_l2'
    _T_PARTIAL = 250
    _ALPHA     = 0.5

    os.makedirs(_L2_OUT, exist_ok=True)

    _betas     = cosine_beta_schedule(1000)
    _diffusion = GaussianDiffusion(_betas, device=DEVICE)
    _feat_ext  = FeatureExtractor().to(DEVICE)

    l2_results = {}

    for _cat in ALL_CATEGORIES:
        _out = Path(_L2_OUT) / 'dit_{}'.format(_cat) / 'evaluation_results.json'
        if _out.exists():
            _d = json.load(open(_out))
            _r = _d.get('results', {}).get(_cat, _d.get(_cat, {}))
            if _r and _r.get('image_auroc', 0) > 0:
                l2_results[_cat] = _r
                print('[SKIP] {:<14} img={:.4f}  pix={:.4f}  (cached)'.format(
                    _cat, _r['image_auroc'], _r['pixel_auroc']))
                continue

        _ckpt = '{}/{}/best.pt'.format(_CKPT_DIR, _cat)
        if not os.path.exists(_ckpt):
            print('[MISS] {}: checkpoint not found'.format(_cat))
            continue

        print('[EVAL] {}...'.format(_cat), end=' ', flush=True)
        _model = DiT_Tiny(img_size=128).to(DEVICE)
        _c = torch.load(_ckpt, map_location=DEVICE, weights_only=False)
        _model.load_state_dict(_c['model_state_dict'])

        _, _tl = get_dataloaders(DATA_ROOT, _cat, img_size=128, batch_size=8)

        _res = evaluate_category(
            _model, _diffusion, _tl, _feat_ext,
            device=DEVICE, t_partial=_T_PARTIAL, num_ddim_steps=50,
            alpha=_ALPHA, img_size=128, scoring='l2',
        )
        l2_results[_cat] = _res
        print('img={:.4f}  pix={:.4f}'.format(_res['image_auroc'], _res['pixel_auroc']))

        _out.parent.mkdir(parents=True, exist_ok=True)
        with open(_out, 'w') as _f:
            json.dump({'scoring': 'l2', 't_partial': _T_PARTIAL,
                       'results': {_cat: _res}}, _f, indent=2)

        del _model
        torch.cuda.empty_cache()

    if l2_results:
        _im = np.mean([v['image_auroc'] for v in l2_results.values()])
        _pm = np.mean([v['pixel_auroc']  for v in l2_results.values()])
        print('\n=== L2 Results ({}/15) ==='.format(len(l2_results)))
        print('Mean Image AUROC: {:.4f}'.format(_im))
        print('Mean Pixel AUROC: {:.4f}'.format(_pm))
        with open('{}/l2_summary.json'.format(_L2_OUT), 'w') as _f:
            json.dump({'scoring': 'l2', 'results': l2_results,
                       'mean_img': float(_im), 'mean_pix': float(_pm)}, _f, indent=2)
        print('Saved: output/results_l2/l2_summary.json')

In [ ]:
import json, numpy as np
from pathlib import Path

# Load L2 results
l2_results = {}
_l2p = Path('output/results_l2/l2_summary.json')
if _l2p.exists():
    l2_results = json.load(open(_l2p))['results']
else:
    print('L2 summary not found. Run Section 2 first.')

# Load prior SSIM results -- check repo path as fallback
ssim_summary = {}
for _sp in ['output/results/all_results_summary.json',
            '../results/dit_ssim/all_results_summary.json']:
    if Path(_sp).exists():
        ssim_summary = json.load(open(_sp))
        print('SSIM baseline loaded from:', _sp)
        break
if not ssim_summary:
    print('SSIM results not found -- SSIM column will show 0.000')

# Print comparison table
print('{:<14} {:>9} {:>9}   {:>9} {:>9}   {:>10}'.format(
    'Category', 'SSIM-Img', 'SSIM-Pix', 'L2-Img', 'L2-Pix', 'Delta-Img'))
print('-' * 70)
for _cat in ALL_CATEGORIES:
    _s = ssim_summary.get(_cat, {}).get('DiT-Tiny', {})
    _l = l2_results.get(_cat, {})
    _si = _s.get('img', 0.0)
    _sp = _s.get('pix', 0.0)
    _li = _l.get('image_auroc', 0.0)
    _lp = _l.get('pixel_auroc', 0.0)
    _d  = _li - _si if _si > 0 else float('nan')
    print('{:<14} {:>9.3f} {:>9.3f}   {:>9.3f} {:>9.3f}   {:>+10.3f}'.format(
        _cat, _si, _sp, _li, _lp, _d))

if l2_results:
    _im = np.mean([v['image_auroc'] for v in l2_results.values()])
    _pm = np.mean([v['pixel_auroc']  for v in l2_results.values()])
    print('-' * 70)
    print('{:<14} {:>9} {:>9}   {:>9.3f} {:>9.3f}'.format('MEAN', '---', '---', _im, _pm))

## Section 3 â€” Ablation: T_partial Sweep

In [ ]:
import json, os, torch, numpy as np
from pathlib import Path

_SWEEP_OUT = 'output/results_l2/ablation_tpartial.json'

if not HAVE_CHECKPOINTS:
    print('No checkpoints. Skipping T_partial ablation.')
elif Path(_SWEEP_OUT).exists():
    print('T_partial sweep cached.')
    _d = json.load(open(_SWEEP_OUT))
    print('{:>10} {:>12} {:>12}'.format('t_partial', 'Image AUROC', 'Pixel AUROC'))
    print('-' * 38)
    for _k, _v in sorted(_d['sweep'].items(), key=lambda x: int(x[0])):
        print('{:>10} {:>12.4f} {:>12.4f}'.format(
            _v['t_partial'], _v['image_auroc'], _v['pixel_auroc']))
else:
    from src.dataset import get_dataloaders
    from src.diffusion import GaussianDiffusion, cosine_beta_schedule
    from src.dit import DiT_Tiny
    from src.scoring import FeatureExtractor
    from src.evaluate import evaluate_category

    _cat   = 'hazelnut'
    _model = DiT_Tiny(img_size=128).to(DEVICE)
    _c = torch.load('output/checkpoints/{}/best.pt'.format(_cat),
                    map_location=DEVICE, weights_only=False)
    _model.load_state_dict(_c['model_state_dict'])
    _model.eval()

    _betas     = cosine_beta_schedule(1000)
    _diffusion = GaussianDiffusion(_betas, device=DEVICE)
    _feat_ext  = FeatureExtractor().to(DEVICE)
    _, _tl = get_dataloaders(DATA_ROOT, _cat, img_size=128, batch_size=8)

    _sweep_data = {'category': _cat, 'scoring': 'l2', 'sweep': {}}
    print('{:>10} {:>12} {:>12}'.format('t_partial', 'Image AUROC', 'Pixel AUROC'))
    print('-' * 38)
    for _t in [100, 150, 200, 250, 300, 400, 500]:
        _res = evaluate_category(
            _model, _diffusion, _tl, _feat_ext,
            device=DEVICE, t_partial=_t, num_ddim_steps=50,
            alpha=0.5, img_size=128, scoring='l2',
        )
        _sweep_data['sweep'][str(_t)] = {
            't_partial': _t,
            'image_auroc': _res['image_auroc'],
            'pixel_auroc': _res['pixel_auroc'],
        }
        print('{:>10} {:>12.4f} {:>12.4f}'.format(_t, _res['image_auroc'], _res['pixel_auroc']))

    os.makedirs(os.path.dirname(_SWEEP_OUT), exist_ok=True)
    with open(_SWEEP_OUT, 'w') as _f:
        json.dump(_sweep_data, _f, indent=2)
    print('Saved:', _SWEEP_OUT)
    del _model
    torch.cuda.empty_cache()

## Section 4 â€” PatchCore Baseline (SOTA Comparison)

In [ ]:
import os, json, torch, numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
from pathlib import Path

# â”€â”€ PatchCoreModel â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
class PatchCoreModel:
    """
    Minimal faithful PatchCore (Roth et al., 2022).
    Backbone: WideResNet50-2 (layers 2+3), random 10% coreset subsampling,
    nearest-neighbour anomaly scoring. Self-contained -- no anomalib needed.
    """

    def __init__(self, device='cuda', subsample=0.1, patchsize=3):
        self.device     = device
        self.subsample  = subsample
        self.patchsize  = patchsize
        self.memory_bank = None
        self._feats      = {}

        try:
            _bb = models.wide_resnet50_2(weights=models.Wide_ResNet50_2_Weights.IMAGENET1K_V1)
        except AttributeError:
            _bb = models.wide_resnet50_2(pretrained=True)
        _bb.eval()
        for _p in _bb.parameters():
            _p.requires_grad = False
        self.backbone = _bb.to(device)

        def _hook(name):
            def _fn(module, inp, out):
                self._feats[name] = out
            return _fn

        self.backbone.layer2.register_forward_hook(_hook('l2'))  # (B,512,16,16)
        self.backbone.layer3.register_forward_hook(_hook('l3'))  # (B,1024,8,8)

        self.norm = transforms.Normalize(
            mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

    @torch.no_grad()
    def _extract(self, images):
        _x = self.norm(((images.to(self.device) + 1.0) / 2.0).clamp(0, 1))
        self.backbone(_x)
        _f2 = self._feats['l2']
        _f3 = F.interpolate(self._feats['l3'], size=_f2.shape[-2:],
                            mode='bilinear', align_corners=False)
        return torch.cat([_f2, _f3], dim=1)   # (B,1536,16,16)

    def _to_patches(self, feat):
        if self.patchsize > 1:
            _p = self.patchsize // 2
            feat = F.avg_pool2d(feat, self.patchsize, stride=1, padding=_p)
        _B, _C, _h, _w = feat.shape
        return feat.permute(0, 2, 3, 1).reshape(-1, _C)   # (B*h*w, C)

    def fit(self, train_loader):
        _patches = []
        for _batch in tqdm(train_loader, desc='  Memory bank', leave=False):
            # Train split returns plain tensor (B,C,H,W); test returns tuple (imgs,masks,labels)
            _imgs = _batch if isinstance(_batch, torch.Tensor) else _batch[0]
            _feat = self._extract(_imgs)
            _patches.append(self._to_patches(_feat).cpu())
        _all = torch.cat(_patches, 0)
        _k = max(1, int(_all.shape[0] * self.subsample))
        _idx = torch.randperm(_all.shape[0])[:_k]
        self.memory_bank = _all[_idx].to(self.device)
        print('  Memory bank: {} patches, dim={}'.format(_k, self.memory_bank.shape[1]))

    @torch.no_grad()
    def predict(self, test_loader, img_size=128):
        _img_scores, _img_labels = [], []
        _pix_preds,  _pix_labels = [], []
        _mb = self.memory_bank

        for _batch in tqdm(test_loader, desc='  Scoring', leave=False):
            _imgs   = _batch[0]
            _masks  = _batch[1]
            _labels = _batch[2]

            _feat = self._extract(_imgs)
            _B, _C, _h, _w = _feat.shape
            _patches = self._to_patches(_feat)   # (B*h*w, C)

            # NN distance to memory bank (chunked to avoid OOM)
            _dists = []
            for _start in range(0, _patches.shape[0], 512):
                _p = _patches[_start:_start + 512]
                _d = (_p.unsqueeze(1) - _mb.unsqueeze(0)).pow(2).sum(-1).min(1).values
                _dists.append(_d.cpu())
            _dmap = torch.cat(_dists).reshape(_B, 1, _h, _w)

            # Upsample to original image size for pixel AUROC
            _pmap = F.interpolate(_dmap.float(), size=(img_size, img_size),
                                  mode='bilinear', align_corners=False)

            # Image score: max anomaly value over spatial map
            _iscore = _pmap.flatten(1).max(dim=1).values

            _img_scores.append(_iscore.numpy())
            _img_labels.append(_labels.numpy())
            _pix_preds.append(_pmap.flatten().numpy())
            _pix_labels.append((_masks.flatten().numpy() > 0.5).astype(int))

        _iscores = np.concatenate(_img_scores)
        _ilabels = np.concatenate(_img_labels)
        _ppreds  = np.concatenate(_pix_preds)
        _plabels = np.concatenate(_pix_labels)

        _img_auroc = float(roc_auc_score(_ilabels, _iscores))
        _pix_auroc = 0.0
        if len(np.unique(_plabels)) > 1:
            _pix_auroc = float(roc_auc_score(_plabels, _ppreds))
        return _img_auroc, _pix_auroc


# â”€â”€ Run PatchCore on all 15 categories â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from src.dataset import get_dataloaders

_PC_OUT = 'output/results_l2/patchcore_all.json'
os.makedirs('output/results_l2', exist_ok=True)

# Load cached results
patchcore_results = {}
if Path(_PC_OUT).exists():
    patchcore_results = json.load(open(_PC_OUT)).get('results', {})
    print('Loaded {} cached PatchCore results.'.format(len(patchcore_results)))

_remaining = [c for c in ALL_CATEGORIES
              if c not in patchcore_results
              or patchcore_results[c].get('image_auroc', 0) == 0]
if not _remaining:
    print('All PatchCore results cached.')
else:
    print('Running PatchCore on {} categories...'.format(len(_remaining)))
    _pc = PatchCoreModel(device=DEVICE, subsample=0.1, patchsize=3)

    for _cat in _remaining:
        print('[{}]'.format(_cat))
        try:
            _tr, _te = get_dataloaders(DATA_ROOT, _cat, img_size=128, batch_size=16)
            _pc.memory_bank = None
            _pc.fit(_tr)
            _ia, _pa = _pc.predict(_te, img_size=128)
            patchcore_results[_cat] = {'image_auroc': _ia, 'pixel_auroc': _pa}
            print('  img={:.4f}  pix={:.4f}'.format(_ia, _pa))
        except Exception as _e:
            print('  ERROR: {}'.format(_e))
            patchcore_results[_cat] = {'image_auroc': 0.0, 'pixel_auroc': 0.0}

        with open(_PC_OUT, 'w') as _f:
            json.dump({'method': 'patchcore_wrn50_10pct_coreset',
                       'results': patchcore_results}, _f, indent=2)

if patchcore_results:
    _im = np.mean([v['image_auroc'] for v in patchcore_results.values()])
    _pm = np.mean([v['pixel_auroc']  for v in patchcore_results.values()])
    print('\nPatchCore ({}/15): Img {:.4f} | Pix {:.4f}'.format(
        len(patchcore_results), _im, _pm))

## Section 5 â€” Report Figures

In [ ]:
import json, numpy as np, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

FIG_DIR = 'output/figures_final'
Path(FIG_DIR).mkdir(parents=True, exist_ok=True)

# â”€â”€ Load all results â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# L2 (new primary)
l2_results = {}
if Path('output/results_l2/l2_summary.json').exists():
    l2_results = json.load(open('output/results_l2/l2_summary.json'))['results']

# PatchCore
patchcore_results = {}
if Path('output/results_l2/patchcore_all.json').exists():
    patchcore_results = json.load(open('output/results_l2/patchcore_all.json'))['results']

# SSIM + classical baselines from committed repo
ssim_summary = {}
for _sp in ['output/results/all_results_summary.json',
            '../results/dit_ssim/all_results_summary.json']:
    if Path(_sp).exists():
        ssim_summary = json.load(open(_sp))
        break

pca_results, ae_results = {}, {}
for _cat in ALL_CATEGORIES:
    _entry = ssim_summary.get(_cat, {})
    if 'PCA' in _entry:
        pca_results[_cat] = {'image_auroc': _entry['PCA']['img'], 'pixel_auroc': _entry['PCA']['pix']}
    if 'Conv-AE' in _entry:
        ae_results[_cat]  = {'image_auroc': _entry['Conv-AE']['img'], 'pixel_auroc': _entry['Conv-AE']['pix']}

print('Results loaded -- L2:{} PC:{} PCA:{} AE:{}'.format(
    len(l2_results), len(patchcore_results), len(pca_results), len(ae_results)))

# â”€â”€ Figure 1: AUROC comparison bar chart â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
_METHODS = [
    ('DiT-Tiny (L2)',  l2_results,          '#2196F3'),
    ('PatchCore',      patchcore_results,   '#4CAF50'),
    ('PCA',            pca_results,         '#FF9800'),
    ('Conv-AE',        ae_results,          '#9C27B0'),
]
_x  = np.arange(len(ALL_CATEGORIES))
_nM = len(_METHODS)
_w  = 0.18

_fig, _axes = plt.subplots(2, 1, figsize=(18, 11))
for _ai, _metric in enumerate(['image_auroc', 'pixel_auroc']):
    _ax = _axes[_ai]
    for _mi, (_name, _data, _col) in enumerate(_METHODS):
        _vals = [_data.get(_c, {}).get(_metric, 0) for _c in ALL_CATEGORIES]
        _bars = _ax.bar(_x + _mi * _w - (_nM - 1) * _w / 2, _vals, _w * 0.88,
                        label=_name, color=_col, alpha=0.85)
        # Dashed mean line
        _v = [_data.get(_c, {}).get(_metric, 0) for _c in ALL_CATEGORIES if _c in _data]
        if _v:
            _ax.axhline(np.mean(_v), color=_col, linestyle=':', linewidth=1.2, alpha=0.7)

    _ax.axhline(0.5, color='red', linestyle='--', linewidth=0.8, alpha=0.4, label='Random')
    _ax.set_xticks(_x)
    _ax.set_xticklabels(ALL_CATEGORIES, rotation=35, ha='right', fontsize=9)
    _ax.set_ylabel('AUROC', fontsize=11)
    _title = 'Image-Level' if _ai == 0 else 'Pixel-Level'
    _ax.set_title(_title + ' AUROC â€” DiT-Tiny (L2) vs Baselines', fontsize=12)
    _ax.set_ylim(0, 1.05)
    _ax.legend(loc='upper right', fontsize=9, ncol=2)
    _ax.grid(axis='y', alpha=0.3)

_fig.suptitle('MVTec AD â€” All 15 Categories (Group 6, DATA-MSML 612)', fontsize=13, fontweight='bold')
plt.tight_layout()
_out = '{}/auroc_comparison.png'.format(FIG_DIR)
plt.savefig(_out, dpi=150, bbox_inches='tight')
plt.close()
print('Saved:', _out)

# â”€â”€ Figure 2: T_partial ablation â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
_tp_path = Path('output/results_l2/ablation_tpartial.json')
if _tp_path.exists():
    _tp = json.load(open(_tp_path))['sweep']
    _tv = sorted([int(k) for k in _tp.keys()])
    _iv = [_tp[str(t)]['image_auroc'] for t in _tv]
    _pv = [_tp[str(t)]['pixel_auroc']  for t in _tv]

    _fig, _ax = plt.subplots(figsize=(8, 5))
    _ax.plot(_tv, _iv, 'o-', color='#2196F3', lw=2, ms=8, label='Image AUROC')
    _ax.plot(_tv, _pv, 's--', color='#FF5722', lw=2, ms=8, label='Pixel AUROC')
    _best_t = _tv[int(np.argmax(_iv))]
    _ax.axvline(_best_t, color='#2196F3', linestyle=':', alpha=0.6,
                label='Best image t={}'.format(_best_t))
    _ax.set_xlabel('t_partial (noise timestep added before reconstruction)', fontsize=11)
    _ax.set_ylabel('AUROC', fontsize=11)
    _ax.set_title('Ablation: t_partial Sweep on Hazelnut (DiT-Tiny, L2 scoring)', fontsize=12)
    _ax.set_ylim(0.0, 1.05)
    _ax.legend(fontsize=10)
    _ax.grid(alpha=0.3)
    plt.tight_layout()
    _out = '{}/ablation_tpartial.png'.format(FIG_DIR)
    plt.savefig(_out, dpi=150, bbox_inches='tight')
    plt.close()
    print('Saved:', _out)
else:
    print('T_partial results not found -- run Section 3 first.')

# â”€â”€ Figure 3: Training loss curves â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import pandas as pd
_cmap = plt.cm.get_cmap('tab20', 15)
_fig, _ax = plt.subplots(figsize=(12, 6))
for _i, _cat in enumerate(ALL_CATEGORIES):
    _csv = 'output/checkpoints/{}/loss.csv'.format(_cat)
    if not Path(_csv).exists():
        _csv = '../results/checkpoints_loss/{}/loss.csv'.format(_cat)
    if Path(_csv).exists():
        _df = pd.read_csv(_csv, header=None, names=['epoch', 'loss', 'lr', 'time'])
        _ax.plot(_df['epoch'], _df['loss'], lw=1.2, color=_cmap(_i), alpha=0.8, label=_cat)
_ax.set_xlabel('Epoch', fontsize=11)
_ax.set_ylabel('Training Loss (diffusion MSE)', fontsize=11)
_ax.set_title('DiT-Tiny Training Convergence â€” All 15 MVTec Categories (100 epochs)', fontsize=12)
_ax.legend(fontsize=7, ncol=4, loc='upper right')
_ax.set_yscale('log')
_ax.grid(alpha=0.3)
plt.tight_layout()
_out = '{}/training_loss_curves.png'.format(FIG_DIR)
plt.savefig(_out, dpi=150, bbox_inches='tight')
plt.close()
print('Saved:', _out)

# â”€â”€ Figure 4: Scoring method ablation (SSIM vs L2 vs LPIPS on hazelnut) â”€â”€â”€â”€â”€â”€
for _sp in ['output/results/ablation_scoring.json',
            '../results/dit_ssim/ablation_scoring.json']:
    if Path(_sp).exists():
        _sd   = json.load(open(_sp))
        _meths = list(_sd['results'].keys())
        _ivals = [_sd['results'][m]['image_auroc'] for m in _meths]
        _pvals = [_sd['results'][m]['pixel_auroc']  for m in _meths]

        _fig, _ax = plt.subplots(figsize=(7, 5))
        _xs = np.arange(len(_meths))
        _b1 = _ax.bar(_xs - 0.2, _ivals, 0.35, label='Image AUROC', color='#2196F3', alpha=0.85)
        _b2 = _ax.bar(_xs + 0.2, _pvals, 0.35, label='Pixel AUROC', color='#FF5722', alpha=0.85)
        for _b, _v in zip(list(_b1) + list(_b2), _ivals + _pvals):
            _ax.text(_b.get_x() + _b.get_width() / 2, _v + 0.01,
                     '{:.3f}'.format(_v), ha='center', va='bottom', fontsize=9)
        _ax.set_xticks(_xs)
        _ax.set_xticklabels([m.upper() for m in _meths], fontsize=12)
        _ax.set_ylim(0, 1.05)
        _ax.set_ylabel('AUROC', fontsize=11)
        _ax.set_title('Pixel Scoring Method Ablation â€” Hazelnut, DiT-Tiny', fontsize=12)
        _ax.axhline(0.5, color='red', linestyle='--', lw=0.8, alpha=0.4, label='Random')
        _ax.legend(fontsize=10)
        _ax.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        _out = '{}/ablation_scoring.png'.format(FIG_DIR)
        plt.savefig(_out, dpi=150, bbox_inches='tight')
        plt.close()
        print('Saved:', _out)
        break

# â”€â”€ Figure 5: 6-panel qualitative visualisations â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import torch
from src.dataset import get_dataloaders
from src.diffusion import GaussianDiffusion, cosine_beta_schedule
from src.dit import DiT_Tiny
from src.scoring import FeatureExtractor, compute_pixel_anomaly_map_l2, compute_feature_anomaly_map, compute_combined_anomaly_map

_VIZ_CATS = ['hazelnut', 'bottle', 'leather', 'screw', 'carpet']
_betas = cosine_beta_schedule(1000)
_diff  = GaussianDiffusion(_betas, device=DEVICE)
_fext  = FeatureExtractor().to(DEVICE)

def _denorm(t):
    return ((t.clamp(-1, 1) + 1) / 2).permute(1, 2, 0).cpu().numpy()

for _cat in _VIZ_CATS:
    _ckpt_path = 'output/checkpoints/{}/best.pt'.format(_cat)
    if not Path(_ckpt_path).exists():
        print('No checkpoint for {} -- skipping viz'.format(_cat))
        continue

    _m = DiT_Tiny(img_size=128).to(DEVICE)
    _c = torch.load(_ckpt_path, map_location=DEVICE, weights_only=False)
    _m.load_state_dict(_c['model_state_dict'])
    _m.eval()

    _, _tl = get_dataloaders(DATA_ROOT, _cat, img_size=128, batch_size=8)

    _anom_img = _norm_img = _anom_mask = None
    with torch.no_grad():
        for _batch in _tl:
            _imgs, _masks, _labs = _batch[0], _batch[1], _batch[2]
            _imgs = _imgs.to(DEVICE)
            _ai   = (_labs == 1).nonzero(as_tuple=True)[0]
            _ni   = (_labs == 0).nonzero(as_tuple=True)[0]
            if len(_ai) > 0 and len(_ni) > 0:
                _anom_img  = _imgs[[_ai[0].item()]]
                _anom_mask = _masks[_ai[0].item()]
                _norm_img  = _imgs[[_ni[0].item()]]
                break

    if _anom_img is None:
        print('Could not find paired images for {}'.format(_cat))
        del _m; torch.cuda.empty_cache(); continue

    with torch.no_grad():
        _pair  = torch.cat([_anom_img, _norm_img], 0)
        _recon = _diff.reconstruct(_m, _pair, t_partial=250, num_ddim_steps=50)
        _l2m   = compute_pixel_anomaly_map_l2(_pair, _recon)
        _fm    = compute_feature_anomaly_map(_fext, _pair, _recon, 128)
        _cm    = compute_combined_anomaly_map(_l2m, _fm, alpha=0.5)

    for _which, _idx, _tag in [('anomalous', 0, 'anom'), ('normal', 1, 'norm')]:
        _panels = [
            _denorm(_pair[_idx]),
            _denorm(_recon[_idx]),
            _l2m[_idx, 0].cpu().numpy(),
            _fm[_idx, 0].cpu().numpy(),
            _cm[_idx, 0].cpu().numpy(),
            _anom_mask.squeeze().cpu().numpy() if _which == 'anomalous' else np.zeros((128, 128)),
        ]
        _titles = ['Original', 'Reconstruction (DiT)', 'L2 Map', 'Feature Map', 'Combined Map', 'GT Mask']
        _fig, _axes = plt.subplots(1, 6, figsize=(18, 3))
        for _ax2, _title, _p in zip(_axes, _titles, _panels):
            if _p.ndim == 3:
                _ax2.imshow(_p)
            else:
                _ax2.imshow(_p, cmap='hot')
            _ax2.set_title(_title, fontsize=9)
            _ax2.axis('off')
        _fig.suptitle('{} -- {} sample'.format(_cat, _which), fontsize=11)
        plt.tight_layout()
        _out = '{}/sixpanel_{}_{}.png'.format(FIG_DIR, _cat, _tag)
        plt.savefig(_out, dpi=120, bbox_inches='tight')
        plt.close()
        print('Saved:', _out)

    del _m; torch.cuda.empty_cache()

print('\nAll figures saved to:', FIG_DIR)

## Section 6 â€” Summary Tables + LaTeX

In [ ]:
import json, numpy as np
from pathlib import Path

# â”€â”€ Load all results â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
l2_results, patchcore_results, ssim_summary = {}, {}, {}

if Path('output/results_l2/l2_summary.json').exists():
    l2_results = json.load(open('output/results_l2/l2_summary.json'))['results']
if Path('output/results_l2/patchcore_all.json').exists():
    patchcore_results = json.load(open('output/results_l2/patchcore_all.json'))['results']
for _sp in ['output/results/all_results_summary.json',
            '../results/dit_ssim/all_results_summary.json']:
    if Path(_sp).exists():
        ssim_summary = json.load(open(_sp)); break

# Build master dict
master = {}
for _cat in ALL_CATEGORIES:
    _s = ssim_summary.get(_cat, {})
    master[_cat] = {}
    if _cat in l2_results:
        master[_cat]['DiT-Tiny (L2)'] = {
            'img': l2_results[_cat]['image_auroc'],
            'pix': l2_results[_cat]['pixel_auroc'],
        }
    if 'DiT-Tiny' in _s:
        master[_cat]['DiT-Tiny (SSIM)'] = _s['DiT-Tiny']
    if _cat in patchcore_results:
        master[_cat]['PatchCore'] = {
            'img': patchcore_results[_cat]['image_auroc'],
            'pix': patchcore_results[_cat]['pixel_auroc'],
        }
    if 'PCA' in _s:
        master[_cat]['PCA']     = _s['PCA']
    if 'Conv-AE' in _s:
        master[_cat]['Conv-AE'] = _s['Conv-AE']

import os
os.makedirs('output/results_l2', exist_ok=True)
with open('output/results_l2/master_results.json', 'w') as _f:
    json.dump(master, _f, indent=2)
print('Saved: output/results_l2/master_results.json')

# â”€â”€ Print table â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
_SHOW = ['DiT-Tiny (L2)', 'PatchCore', 'PCA', 'Conv-AE']
print()
_hdr = '{:<14}'.format('Category')
for _m in _SHOW:
    _hdr += '  {:>8} {:>8}'.format(_m[:5], 'Pix')
print(_hdr)
print('-' * (14 + len(_SHOW) * 20))
for _cat in ALL_CATEGORIES:
    _row = '{:<14}'.format(_cat)
    for _m in _SHOW:
        _d = master[_cat].get(_m, {})
        _row += '  {:>8.3f} {:>8.3f}'.format(_d.get('img', 0), _d.get('pix', 0))
    print(_row)
print('-' * (14 + len(_SHOW) * 20))
_mrow = '{:<14}'.format('MEAN')
for _m in _SHOW:
    _iv = [master[c].get(_m, {}).get('img', 0) for c in ALL_CATEGORIES if _m in master.get(c, {})]
    _pv = [master[c].get(_m, {}).get('pix', 0) for c in ALL_CATEGORIES if _m in master.get(c, {})]
    _mrow += '  {:>8.3f} {:>8.3f}'.format(np.mean(_iv) if _iv else 0, np.mean(_pv) if _pv else 0)
print(_mrow)

# â”€â”€ LaTeX table â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
_bs = chr(92)
_lines = [
    _bs + 'begin{table}[t]',
    _bs + 'centering',
    _bs + 'caption{Image- and pixel-level AUROC on MVTec AD (15 categories). '
          'DiT-Tiny trained 100 epochs per category on Colab T4 GPU. '
          'PatchCore uses pretrained WideResNet50-2 with 10' + _bs + '% random coreset.}',
    _bs + 'label{tab:main_results}',
    _bs + 'resizebox{' + _bs + 'textwidth}{!}{%',
    _bs + 'begin{tabular}{l' + 'rr' * len(_SHOW) + '}',
    _bs + 'toprule',
]
_h1 = 'Category'
for _m in _SHOW:
    _h1 += ' & ' + _bs + 'multicolumn{2}{c}{' + _m.replace('(', '(').replace(')', ')') + '}'
_lines.append(_h1 + ' ' + _bs * 2)
_h2 = ''
for _ in _SHOW:
    _h2 += ' & Img & Pix'
_lines.append(_h2 + ' ' + _bs * 2)
_lines.append(_bs + 'midrule')
for _cat in ALL_CATEGORIES:
    _row = _cat.replace('_', _bs + '_')
    for _m in _SHOW:
        _d = master[_cat].get(_m, {})
        _row += ' & {:.3f} & {:.3f}'.format(_d.get('img', 0), _d.get('pix', 0))
    _lines.append(_row + ' ' + _bs * 2)
_lines.append(_bs + 'midrule')
_mr = _bs + 'textbf{Mean}'
for _m in _SHOW:
    _iv = [master[c].get(_m, {}).get('img', 0) for c in ALL_CATEGORIES if _m in master.get(c, {})]
    _pv = [master[c].get(_m, {}).get('pix', 0) for c in ALL_CATEGORIES if _m in master.get(c, {})]
    if _iv:
        _mr += (' & ' + _bs + 'textbf{' + '{:.3f}'.format(np.mean(_iv)) + '}' +
                ' & ' + _bs + 'textbf{' + '{:.3f}'.format(np.mean(_pv)) + '}')
    else:
        _mr += ' & --- & ---'
_lines.append(_mr + ' ' + _bs * 2)
_lines += [_bs + 'bottomrule', _bs + 'end{tabular}%', '}', _bs + 'end{table}']
_tex = chr(10).join(_lines)
with open('output/results_l2/table_main_results.tex', 'w') as _f:
    _f.write(_tex)
print('\nLaTeX table saved: output/results_l2/table_main_results.tex')
print()
print(_tex)

## Section 7 â€” Package to Drive + Verify

In [ ]:
import json, zipfile, os, shutil
from pathlib import Path
from datetime import datetime

DRIVE_DIR = '/content/drive/MyDrive'
_ts  = datetime.now().strftime('%Y%m%d_%H%M')
_zip = '/tmp/FINAL_RESULTS_{}.zip'.format(_ts)

_INCLUDE = [
    ('output/results_l2',    lambda p: True),                    # all new results
    ('output/figures_final', lambda p: True),                    # all figures
    ('output/results',       lambda p: p.suffix != '.pt'),       # prior SSIM results
    ('output/checkpoints',   lambda p: p.name == 'loss.csv'),    # loss CSVs only
]

print('Creating {}...'.format(os.path.basename(_zip)))
with zipfile.ZipFile(_zip, 'w', zipfile.ZIP_DEFLATED) as _zf:
    for _folder, _filt in _INCLUDE:
        if not os.path.exists(_folder):
            print('  (missing, skipping):', _folder)
            continue
        for _fp in Path(_folder).rglob('*'):
            if _fp.is_file() and _filt(_fp):
                _zf.write(_fp, str(_fp.relative_to('.')))

_sz = os.path.getsize(_zip) / 1e6
print('Zip: {:.1f} MB'.format(_sz))

_dp = os.path.join(DRIVE_DIR, 'FINAL_RESULTS_{}.zip'.format(_ts))
shutil.copy2(_zip, _dp)
shutil.copy2(_zip, os.path.join(DRIVE_DIR, 'FINAL_RESULTS_LATEST.zip'))
print('Saved to Drive:', _dp)
print('Updated:        FINAL_RESULTS_LATEST.zip')

from google.colab import files
files.download(_zip)
print('Download triggered.')

# â”€â”€ Verification checklist â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print('\n=== VERIFICATION CHECKLIST ===')
_checks = [
    ('L2 summary (15 cats)',     'output/results_l2/l2_summary.json',       'results', 15),
    ('PatchCore (15 cats)',      'output/results_l2/patchcore_all.json',     'results', 15),
    ('T_partial ablation (7)',   'output/results_l2/ablation_tpartial.json', 'sweep',    7),
    ('Master results JSON',      'output/results_l2/master_results.json',    None,       1),
    ('LaTeX table',              'output/results_l2/table_main_results.tex', None,       1),
    ('AUROC bar chart',          'output/figures_final/auroc_comparison.png', None,      1),
    ('T_partial plot',           'output/figures_final/ablation_tpartial.png', None,     1),
    ('Loss curves',              'output/figures_final/training_loss_curves.png', None,  1),
    ('Scoring ablation fig',     'output/figures_final/ablation_scoring.png', None,      1),
]
_pass = 0
for _label, _path, _key, _exp in _checks:
    _p = Path(_path)
    if not _p.exists():
        print('  FAIL  {}: file not found'.format(_label))
        continue
    if _key:
        try:
            _n = len(json.load(open(_p))[_key])
            _ok = _n >= _exp
            print('  {}  {} ({}/{})'.format('PASS' if _ok else 'WARN', _label, _n, _exp))
            if _ok: _pass += 1
        except Exception as _e:
            print('  FAIL  {}: {}'.format(_label, _e))
    else:
        print('  PASS  {}'.format(_label))
        _pass += 1
print('{}/{} checks passed.'.format(_pass, len(_checks)))